In [50]:
# ============================================================
# DEFINICIONES DE ESTADOS – PLEGADORA (alineada a PLE-7)
# ============================================================
#
# APAGADO:
#   - Representa apagado real (corriente ≈ 0)
#   - Condición:
#       estado_R ∈ {2, 3}
#       estado_S ∈ {2, 3}
#       estado_T == 2
#   - Uso:
#       * Corte duro de eventos
#       * Delimitación de inicio/fin de día
#       * Filtro post-evento (eventos de apagado)
#
# REPOSO OPERATIVO:
#   - Máquina encendida sin trabajo
#   - Condición:
#       estado_R == 1
#       estado_S == 1
#       estado_T == 1
#   - Uso:
#       * Punto lógico de inicio de evento
#       * Cierre normal de eventos
#       * Separación entre operaciones
#
# TRABAJO EFECTIVO:
#   - Trabajo real de la plegadora
#   - Condición (consenso 2 de 3 fases):
#       estado_R == 0
#       estado_S == 0
#       estado_T ∈ {0, 3}
#       → se considera trabajo si al menos 2 fases cumplen
#   - Uso:
#       * Inicio de eventos
#       * Continuidad del evento
#   - Nota:
#       * Evita perder picos por desfasaje entre fases
#
# ============================================================
# PARÁMETROS CONFIGURABLES – DETECCIÓN DE EVENTOS
# ============================================================
#
# N_ESTABLE:
#   - Cantidad mínima de muestras consecutivas en estado estable
#     (reposo_operativo o apagado)
#   - Uso:
#       * Determinar comienzo real del día
#       * Determinar fin real del día
#   - Valores típicos:
#       * Muestreo 1 s  → 60–120
#       * Muestreo 5 s  → 12–24
#
# PAUSA_MIN:
#   - Cantidad mínima de reposo_operativo consecutivo
#   - Uso:
#       * Cierre normal de eventos
#       * Filtrado de comienzos de ciclo (persistencia mínima)
#   - Valores típicos:
#       * 40–90 s para plegadora pesada
#
# VENTANA_DESCARTE_APAGADO:
#   - Tiempo posterior al evento durante el cual, si el estado es
#     apagado sostenido, el evento se descarta
#   - Uso:
#       * Eliminación de eventos de fin de jornada
#   - Valores típicos:
#       * 120–300 s (180 s recomendado)
#
# VENTANA_PICO_INICIO (opcional):
#   - Ventana hacia atrás para ajustar el inicio del evento al último
#     pico eléctrico previo
#   - Uso:
#       * Refinamiento temporal del inicio
#       * No afecta la detección de eventos
#
# ============================================================
# NOTA GENERAL:
#   - No agregar nuevos parámetros salvo necesidad física comprobada
#   - Si algo falla:
#       * Primero revisar definición de TRABAJO
#       * Luego calibrar PAUSA_MIN
# ============================================================


In [51]:
# Librerias

import numpy as np
import pandas as pd
import sklearn
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go

In [52]:
# Leer rutas

from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
DATA_PATH = PROJECT_ROOT / "data"

#print("PROJECT_ROOT:", PROJECT_ROOT)
#print("DATA_PATH:", DATA_PATH)
#print("Existe data:", DATA_PATH.exists())

In [53]:
# Abrir los graficos en el navegador

pio.renderers.default = 'browser'

In [54]:
# Funcion para cargar datos

from pathlib import Path
import pandas as pd

def cargar_maquina(base_path, maquina):
    base_path = Path(base_path).resolve()
    paths = list((base_path / maquina).rglob("*.csv"))

    if not paths:
        raise ValueError(f"No se encontraron CSV para {maquina} en {base_path}")

    dfs = []
    for path in paths:
        df = pd.read_csv(path)
        df["maquina"] = maquina
        dfs.append(df)

    return (
        pd.concat(dfs, ignore_index=True)
          .sort_values("temporal_placa")
          .reset_index(drop=True)
    )

In [55]:
# Carga de datos

df_ple1 = cargar_maquina(DATA_PATH, "PLE1")

In [56]:
# Funcion preparar_df

def preparar_df(df):
    df = df.copy()

    # Timestamp
    df['temporal_placa'] = pd.to_datetime(df['temporal_placa'])
    df['hora'] = df['temporal_placa'].dt.hour
    df['minuto'] = df['temporal_placa'].dt.minute

    # Turnos
    def asignar_turno(hora, minuto):
        t = hora * 60 + minuto

        # Pausas
        if 12*60 <= t < 12*60 + 30:
            return 'ALMUERZO'
        if 22*60 <= t < 22*60 + 30:
            return 'CENA'

        # Turnos
        if 5*60 <= t < 17*60:
            return 'TURNO MAÑANA'
        if 17*60 <= t < 22*60:
            return 'TURNO TARDE'
        if (t >= 22*60 + 30) or (t < 1*60):
            return 'TURNO TARDE'

        return 'FUERA_TURNO'

    df['turno'] = df.apply(
        lambda x: asignar_turno(x['hora'], x['minuto']),
        axis=1
    )
        
    # Potencias totales por timestamp
    df['p_activa_total'] = (
        df['potencia_a_r'] +
        df['potencia_a_s'] +
        df['potencia_a_t']
    )

    df['q_reactiva_total'] = (
        df['potencia_r_r'] +
        df['potencia_r_s'] +
        df['potencia_r_t']
    )

    return df


In [57]:
# Ajuste de datos de PLE1

df_ple1 = preparar_df(df_ple1)

df_ple1['temporal_placa'] = (
    pd.to_datetime(df_ple1['temporal_placa'], errors='coerce')
    .dt.tz_localize(None)
)


In [58]:
# Ventanas para PLE1

import numpy as np
import pandas as pd

win = 5  # muestras (si tu muestreo es ~1 Hz, equivale a 5 s)

rows = []

df_ple1['temporal_placa'] = pd.to_datetime(df_ple1['temporal_placa'], errors='coerce').dt.tz_localize(None)
df_ple1 = df_ple1.reset_index(drop=True)

fases = {
    'R': {'I': 'corriente_r', 'P': 'potencia_a_r'},
    'S': {'I': 'corriente_s', 'P': 'potencia_a_s'},
    'T': {'I': 'corriente_t', 'P': 'potencia_a_t'},
}

for fase, cols in fases.items():
    col_I = cols['I']
    col_P = cols['P']

    for i in range(0, len(df_ple1) - win, win):
        segI = df_ple1.iloc[i:i+win][col_I].astype(float).values
        segP = df_ple1.iloc[i:i+win][col_P].astype(float).values
        segT = df_ple1.iloc[i:i+win]['temporal_placa']

        # dt por muestra (robusto ante muestreo irregular)
        dt = segT.diff().dt.total_seconds().fillna(0).values

        # energía de ventana basada en potencia activa (J si P en W)
        energia = np.sum(segP * dt)

        rows.append({
            't_inicio': df_ple1.iloc[i]['temporal_placa'],
            'fase': fase,
            'media': np.mean(segI),
            'std': np.std(segI),
            'pendiente': segI[-1] - segI[0],
            'energia': energia
        })

df_feat = pd.DataFrame(rows)

df_R = df_feat[df_feat['fase'] == 'R'].reset_index(drop=True)
df_S = df_feat[df_feat['fase'] == 'S'].reset_index(drop=True)
df_T = df_feat[df_feat['fase'] == 'T'].reset_index(drop=True)


In [59]:
# Normalizacion por fase

from sklearn.preprocessing import StandardScaler

features = ['media', 'std', 'pendiente', 'energia']

scaler_R = StandardScaler()
Xn_R = scaler_R.fit_transform(df_R[features])

scaler_S = StandardScaler()
Xn_S = scaler_S.fit_transform(df_S[features])

scaler_T = StandardScaler()
Xn_T = scaler_T.fit_transform(df_T[features])



In [60]:
# Clustering por fase

from sklearn.cluster import KMeans

k_R = KMeans(n_clusters=4, random_state=42)
df_R['estado'] = k_R.fit_predict(Xn_R)

k_S = KMeans(n_clusters=4, random_state=42)
df_S['estado'] = k_S.fit_predict(Xn_S)

k_T = KMeans(n_clusters=4, random_state=42)
df_T['estado'] = k_T.fit_predict(Xn_T)



In [61]:
# Estados Fase R PLE1

df_R.groupby('estado')[features].mean()


,media,std,pendiente,energia
estado,,,,
0,18.417682,2.860476,-0.843059,1.718579e+04
1,16.003463,0.129570,0.072086,6.003975e+03
2,0.685873,0.001165,-0.003102,1.196174e+07
3,0.706521,0.046931,0.105123,2.949146e+04


In [62]:
# Estados Fase S PLE1

df_S.groupby('estado')[features].mean()


,media,std,pendiente,energia
estado,,,,
0,19.177262,2.824905,-0.749983,1.803510e+04
1,16.864754,0.140583,0.063997,6.659445e+03
2,0.672904,0.000974,-0.002791,2.298116e+07
3,0.671754,0.032249,0.064734,5.600534e+04


In [63]:
# Estados Fase T PLE1

df_T.groupby('estado')[features].mean()


,media,std,pendiente,energia
estado,,,,
0,18.779290,2.574039,5.419030,29158.461711
1,16.424709,0.129181,0.040942,7785.030531
2,0.152509,0.011450,0.023745,9893.131821
3,18.763528,2.921134,-3.516812,17239.007230


In [64]:
# Nombramiento de estados

df_states = (
    df_R[['t_inicio', 'estado']]
    .rename(columns={'estado': 'estado_R'})
    .merge(
        df_S[['t_inicio', 'estado']].rename(columns={'estado': 'estado_S'}),
        on='t_inicio'
    )
    .merge(
        df_T[['t_inicio', 'estado']].rename(columns={'estado': 'estado_T'}),
        on='t_inicio'
    )
)

# Apagado real
df_states['apagado'] = (
    (df_states['estado_R'].isin([2, 3])) &
    (df_states['estado_S'].isin([2, 3])) &
    (df_states['estado_T'] == 2)
)

# Reposo operativo (encendida sin trabajar)
df_states['reposo_operativo'] = (
    (df_states['estado_R'] == 1) &
    (df_states['estado_S'] == 1) &
    (df_states['estado_T'] == 1)
)

# Trabajo efectivo
df_states['trabajo'] = (
    (
        (df_states['estado_R'] == 0).astype(int) +
        (df_states['estado_S'] == 0).astype(int) +
        (df_states['estado_T'].isin([0, 3])).astype(int)
    ) >= 2
)



In [65]:
PAUSA_MIN = 13      # ventanas consecutivas de reposo para cerrar evento
N_ESTABLE = 16      # ventanas consecutivas de reposo para validar zona
umbral_pico_ple1 = 0.4

In [66]:
# Determinar comienzo y fin reales de los datos

# -------- INICIO VÁLIDO --------
contador = 0
idx_inicio_valido = None

for i, row in df_states.iterrows():
    estado_estable = row['reposo_operativo'] or row['apagado']
    if estado_estable:
        contador += 1
        if contador >= N_ESTABLE:
            idx_inicio_valido = i
            break
    else:
        contador = 0

# -------- FIN VÁLIDO --------
contador = 0
idx_fin_valido = None

for i in range(len(df_states) - 1, -1, -1):
    row = df_states.loc[i]
    estado_estable = row['reposo_operativo'] or row['apagado']
    if estado_estable:
        contador += 1
        if contador >= N_ESTABLE:
            idx_fin_valido = i
            break
    else:
        contador = 0

# Fallbacks de seguridad
if idx_inicio_valido is None:
    idx_inicio_valido = df_states.index.min()

if idx_fin_valido is None:
    idx_fin_valido = df_states.index.max()

if idx_inicio_valido >= idx_fin_valido:
    raise ValueError("No se pudo determinar una zona válida de operación")

t_inicio_valido = df_states.loc[idx_inicio_valido, 't_inicio']
t_fin_valido    = df_states.loc[idx_fin_valido, 't_inicio']


In [67]:
# Funcion detectar_picos mediante corriente

def detectar_picos_i(signal, timestamp, umbral_pico, ventana):

    dt = timestamp.diff().dt.total_seconds()
    valid = dt > 0

    dI_dt = signal.diff() / dt

    # subida fuerte
    subida = valid & dI_dt.notna() & (dI_dt > umbral_pico)

    picos = subida.copy() * False

    # buscar máximo local después de la subida
    for i in range(1, len(signal) - ventana):
        if subida.iloc[i]:
            entorno = signal.iloc[i:i+ventana+1]
            if signal.iloc[i+1] == entorno.max():
                picos.iloc[i+1] = True

    return picos


In [68]:
# Detección de picos en las 3 fases de la PLE1

import pandas as pd

# -----------------------------
# Deteccion de picos en corriente por fase
# -----------------------------
mask_picos_Ir = detectar_picos_i(
    df_ple1['corriente_r'],
    df_ple1['temporal_placa'],
    umbral_pico = umbral_pico_ple1,
    ventana=2
)

mask_picos_Is = detectar_picos_i(
    df_ple1['corriente_s'],
    df_ple1['temporal_placa'],
    umbral_pico = umbral_pico_ple1,
    ventana=2
)

mask_picos_It = detectar_picos_i(
    df_ple1['corriente_t'],
    df_ple1['temporal_placa'],
    umbral_pico = umbral_pico_ple1,
    ventana=2
)

df_ple1['pico_Ir'] = mask_picos_Ir
df_ple1['pico_Is'] = mask_picos_Is
df_ple1['pico_It'] = mask_picos_It

# -----------------------------
# DataFrame de picos
# -----------------------------
mask_picos_final = (
    df_ple1['pico_Ir'] |
    df_ple1['pico_Is'] |
    df_ple1['pico_It']
)

df_peaks_all_ple1 = (
    df_ple1.loc[
        mask_picos_final,
        ['temporal_placa', 'pico_Ir', 'pico_Is', 'pico_It']
    ]
    .rename(columns={'temporal_placa': 't'})
    .reset_index(drop=True)
)

def origen_pico(row):
    origenes = []
    if row['pico_Ir']:
        origenes.append('Ir')
    if row['pico_Is']:
        origenes.append('Is')
    if row['pico_It']:
        origenes.append('It')
    return '+'.join(origenes)

df_peaks_all_ple1['origen'] = df_peaks_all_ple1.apply(origen_pico, axis=1)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
df_peaks_all_ple1

,t,pico_Ir,pico_Is,pico_It,origen
0,2026-01-09 16:57:39,True,True,True,Ir+Is+It
1,2026-01-09 16:57:49,True,True,True,Ir+Is+It
2,2026-01-09 17:04:56,True,True,True,Ir+Is+It
3,2026-01-09 17:05:06,True,True,True,Ir+Is+It
4,2026-01-09 17:11:30,True,True,True,Ir+Is+It
5,2026-01-09 17:16:07,False,False,True,It
6,2026-01-09 18:10:05,True,True,True,Ir+Is+It
7,2026-01-09 18:18:27,True,True,True,Ir+Is+It
8,2026-01-09 18:20:44,True,True,True,Ir+Is+It
9,2026-01-09 18:29:01,True,True,True,Ir+Is+It


In [69]:
# Definicion EVENTO

eventos = []

en_evento = False
inicio = None
contador_reposo = 0
ultimo_reposo = None

for i in range(idx_inicio_valido, idx_fin_valido + 1):

    row = df_states.loc[i]

    apagado = row['apagado']
    reposo = row['reposo_operativo']
    trabajo = row['trabajo']

    # -----------------------------
    # Apagado: cortar evento sí o sí
    # -----------------------------
    if apagado:
        if en_evento:
            eventos.append({
                'Fecha Inicio': inicio,
                'Fecha Fin': row['t_inicio']
            })
            en_evento = False
            inicio = None
        contador_reposo = 0
        ultimo_reposo = None
        continue

    # -----------------------------
    # Guardar último reposo SOLO fuera de evento
    # -----------------------------
    if reposo and not en_evento:
        ultimo_reposo = row['t_inicio']

    # -----------------------------
    # INICIO DE EVENTO
    # -----------------------------
    if trabajo and not en_evento:
        inicio = ultimo_reposo if ultimo_reposo is not None else row['t_inicio']
        en_evento = True
        contador_reposo = 0

    # -----------------------------
    # CONTINUIDAD
    # -----------------------------
    if en_evento and not reposo:
        contador_reposo = 0

    # -----------------------------
    # CIERRE NORMAL
    # -----------------------------
    if en_evento and reposo:
        contador_reposo += 1
        if contador_reposo >= PAUSA_MIN:
            fin = df_states.loc[i - PAUSA_MIN, 't_inicio']
            eventos.append({
                'Fecha Inicio': inicio,
                'Fecha Fin': fin
            })
            en_evento = False
            inicio = None
            contador_reposo = 0
            ultimo_reposo = row['t_inicio']

# -------- cerrar evento abierto --------
if en_evento:
    eventos.append({
        'Fecha Inicio': inicio,
        'Fecha Fin': df_states.loc[idx_fin_valido, 't_inicio']
    })

df_eventos = pd.DataFrame(eventos)


In [70]:
# Correccion del comienzo de los eventos mediante la deteccion de picos

VENTANA_PICO_INICIO = 100  # segundos

df_eventos = df_eventos.copy()
df_eventos['Fecha Inicio Ajustada'] = df_eventos['Fecha Inicio']

for i, ev in df_eventos.iterrows():

    t_ini = ev['Fecha Inicio']
    t_min = t_ini - pd.Timedelta(seconds=VENTANA_PICO_INICIO)

    # picos cercanos antes del evento
    picos_previos = df_peaks_all_ple1[
        (df_peaks_all_ple1['t'] >= t_min) &
        (df_peaks_all_ple1['t'] < t_ini)
    ]

    if not picos_previos.empty:
        # tomamos el último pico antes del evento
        t_pico = picos_previos['t'].max()
        df_eventos.at[i, 'Fecha Inicio Ajustada'] = t_pico



In [71]:
# Deteccion de valles dentro de eventos

VALLE_MIN = 1
VALLE_MAX = 10

valles = []

def referencia_evento(df_signal, t_ini, t_fin, col_I):
    seg = df_signal[
        (df_signal['temporal_placa'] >= t_ini) &
        (df_signal['temporal_placa'] <= t_fin)
    ][col_I]

    media = seg.mean()
    p25 = seg.quantile(0.25)

    return media, p25


for _, ev in df_eventos.iterrows():

    t_ini = ev['Fecha Inicio Ajustada']
    t_fin = ev['Fecha Fin']

    # referencia del evento
    media_evt, p25_evt = referencia_evento(
        df_ple1,
        t_ini,
        t_fin,
        col_I='corriente_r'
    )

    seg = df_feat[
        (df_feat['fase'] == 'R') &
        (df_feat['t_inicio'] >= t_ini) &
        (df_feat['t_inicio'] <= t_fin)
    ]

    contador = 0
    t_valle = None

    for _, row in seg.iterrows():

        es_valle = (
            row['media'] <= p25_evt * 2 and
            row['media'] >= media_evt * 0.9
        )

        if es_valle:
            if contador == 0:
                t_valle = row['t_inicio']
            contador += 1
        else:
            if VALLE_MIN <= contador <= VALLE_MAX:
                valles.append({
                    't_valle': t_valle
                })
            contador = 0
            t_valle = None



In [72]:
# Graficas de eventos para PLE1

import plotly.graph_objects as go

maq = 'PLE1'

map_fase_col = {
    'R': 'corriente_r',
    'S': 'corriente_s',
    'T': 'corriente_t'
}

colores_fase = {
    'R': 'red',
    'S': 'green',
    'T': 'blue'
}

fig = go.Figure()
trace_idx = {}

# Trazas de corriente
for fase, col in map_fase_col.items():

    visible = (fase == 'R')

    fig.add_trace(
        go.Scatter(
            x=df_ple1['temporal_placa'],
            y=df_ple1[col],
            mode='lines',
            name=f'Fase {fase}',
            line=dict(color=colores_fase[fase]),
            visible=visible
        )
    )

    trace_idx[fase] = len(fig.data) - 1

# Shapes de eventos (comunes a todas las fases)
shapes_eventos = []

for _, ev in df_eventos.iterrows():
    shapes_eventos.append(
        dict(
            type='rect',
            xref='x',
            yref='paper',
            x0=ev['Fecha Inicio Ajustada'],
            x1=ev['Fecha Fin'],
            y0=0,
            y1=1,
            fillcolor='grey',
            opacity=0.2,
            line_width=0
        )
    )

# Botones por fase
botones = []

for fase in map_fase_col.keys():

    visibles = [False] * len(fig.data)
    visibles[trace_idx[fase]] = True

    botones.append(
        dict(
            label=f'Fase {fase}',
            method='update',
            args=[
                {'visible': visibles},
                {'shapes': shapes_eventos}
            ]
        )
    )

fig.update_layout(
    title='PLE1 – Corriente por fase con eventos de máquina',
    xaxis_title='Tiempo',
    yaxis_title='Corriente [A]',
    template='plotly_white',
    shapes=shapes_eventos,
    updatemenus=[
        dict(
            buttons=botones,
            direction='down',
            x=1.08,
            y=1.1,
            showactive=True
        )
    ]
)

fig.show()


In [73]:
# Eventos a .cvs
from pathlib import Path

OUT_TABLAS = PROJECT_ROOT / "output" / "tablas"
OUT_TABLAS.mkdir(parents=True, exist_ok=True)

df_eventos.to_csv(OUT_TABLAS / "df_eventos_ple1.csv", index=False)